# Hitter Future Projections - Detailed Workflow

Complete workflow for generating multi-year hitter WAR projections.

**Purpose:** Hitter-specific projection pipeline with detailed control and analysis.

**Features:**
- Position-specific aging curves
- Hitter model features (K%, BB%, AVG, OBP, SLG, etc.)
- Elite hitter protection
- Baserunning and defense adjustments

In [ ]:
# Cell 1: Imports and Setup

import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to path
project_root = Path('.').absolute().parent.parent
sys.path.insert(0, str(project_root))

# Import hitter-specific modules
from new_pipeline.models.future_season import (
    FutureProjectionPipeline,
    load_historical_player_data,
    build_longitudinal_sequences
)
from new_pipeline.common.constants import HITTER_MODEL_FEATURES

print("Hitter Future Projections - sWARm")
print("=" * 70)
print("Multi-year WAR projections for position players")
print()

# Configuration
BASE_YEAR = 2024
YEARS_AHEAD = 3

print(f"Configuration:")
print(f"  Player type: Hitters")
print(f"  Base year: {BASE_YEAR}")
print(f"  Projection years: {BASE_YEAR + 1} - {BASE_YEAR + YEARS_AHEAD}")
print(f"  Model features: {len(HITTER_MODEL_FEATURES)}")
print(f"    {', '.join(HITTER_MODEL_FEATURES)}")

## Step 1: Data Loading and Preparation

In [ ]:
# Cell 2: Load Historical Hitter Data

print("\nLoading historical hitter data...")

historical_data = load_historical_player_data(
    player_type='hitter',
    years=list(range(BASE_YEAR - 8, BASE_YEAR + 1))
)

print(f"  Loaded {len(historical_data)} hitter-season records")
print(f"  Years: {historical_data['Year'].min()} - {historical_data['Year'].max()}")
print(f"  Unique hitters: {historical_data['playerid'].nunique()}")

# Show position distribution
print("\nPosition distribution:")
if 'Position' in historical_data.columns or 'primary_position' in historical_data.columns:
    pos_col = 'Position' if 'Position' in historical_data.columns else 'primary_position'
    pos_counts = historical_data[pos_col].value_counts()
    for pos, count in pos_counts.head(10).items():
        print(f"  {pos}: {count}")

In [ ]:
# Cell 3: Build Longitudinal Sequences

print("\nBuilding longitudinal sequences for hitters...")

sequences_df = build_longitudinal_sequences(
    historical_data,
    player_type='hitter'
)

print(f"  Created {len(sequences_df)} sequences")
print(f"  Sequence years: {sequences_df['year_n'].min()} - {sequences_df['year_n'].max()}")
print(f"  Features per sequence: {len(sequences_df.columns)}")

# Show feature summary
print("\nKey features included:")
feature_cols = [col for col in sequences_df.columns if col.endswith('_n')]
print(f"  Total: {len(feature_cols)} features")
print(f"  Sample: {', '.join(feature_cols[:10])}...")

## Step 2: Initialize and Run Projection Pipeline

In [ ]:
# Cell 4: Initialize Hitter Pipeline

print("\nInitializing hitter projection pipeline...")

hitter_pipeline = FutureProjectionPipeline(
    player_type='hitter',
    base_year=BASE_YEAR,
    years_ahead=YEARS_AHEAD
)

print("\nPipeline components initialized:")
print("  [*] Longitudinal Model - RandomForest with 9 hitter features")
print("  [*] Survival Model - Cox PH for retirement probability")
print("  [*] Age Curves - Position-specific aging (C, SS, 2B, 3B, 1B, OF, DH)")
print("  [*] Elite Adjustments - Protect 6+ WAR hitters from over-regression")
print("  [*] Injury Recovery - ACL and Tommy John adjustments")

In [ ]:
# Cell 5: Run Full Pipeline

print("\nRunning complete hitter projection pipeline...")
print("This includes: data loading, model training, projection generation, and adjustments.")
print()

hitter_projections = hitter_pipeline.run_full_pipeline(
    injury_records=None,  # Optional: provide injury data if available
    save_output=True
)

print(f"\nHitter projections complete!")
print(f"  Total hitters projected: {len(hitter_projections)}")

In [ ]:
# Cell 7.5: Ensemble Model Breakdown

print("\n" + "=" * 70)
print("ENSEMBLE MODEL BREAKDOWN")
print("=" * 70)

# Access the ensemble model from the pipeline
ensemble_model = hitter_pipeline.longitudinal_model

# Display ensemble configuration
print("\nEnsemble Configuration:")
print("-" * 70)
print("Adaptive Weighting by Player History Length:")
print("  Veterans (5+ years):     XGB=0.35, RNN=0.35, ExtraTrees=0.30")
print("  Mid-career (3-4 years):  XGB=0.45, RNN=0.25, ExtraTrees=0.30")
print("  Short history (<4 years): Fallback model (ExtraTrees)")

# Count players by history length from historical data
print("\nPlayer Distribution by History Length:")
print("-" * 70)

player_history_lengths = historical_data.groupby('playerid')['Year'].count()
veterans = (player_history_lengths >= 5).sum()
mid_career = ((player_history_lengths >= 3) & (player_history_lengths < 5)).sum()
short_history = (player_history_lengths < 3).sum()

total_players = len(player_history_lengths)
print(f"  Veterans (5+ seasons):    {veterans:4d} ({100*veterans/total_players:5.1f}%)")
print(f"  Mid-career (3-4 seasons): {mid_career:4d} ({100*mid_career/total_players:5.1f}%)")
print(f"  Short history (<3):       {short_history:4d} ({100*short_history/total_players:5.1f}%)")
print(f"  Total players:            {total_players:4d}")

print("\nModel Architecture:")
print("-" * 70)
print("  [1] XGBoost (Darts XGBModel)")
print("      - Gradient boosting with automatic lag creation")
print("      - Uses last 3 years of WAR + features")
print("      - Best for players with stable performance")
print()
print("  [2] RNN (Darts GRU)")
print("      - Learns WAR trajectory patterns")
print("      - Requires 3+ consecutive seasons")
print("      - Best for capturing performance trends")
print()
print("  [3] ExtraTrees (Darts SKLearnModel)")
print("      - Ensemble of decision trees")
print("      - Uses last 3 years of features")
print("      - Robust to outliers")
print()
print("  [4] ExtraTrees Fallback")
print("      - Standard RandomForest")
print("      - For players with <4 seasons or non-consecutive careers")
print("      - Trained on all players (cross-player learning)")

print("\nEnsemble Benefits:")
print("-" * 70)
print("  - Combines strengths of multiple algorithms")
print("  - Adaptive weighting based on player history")
print("  - Robust to career gaps and injuries")
print("  - No players filtered out (fallback handles all cases)")
print()
print("=" * 70)

## Step 3: Analyze Projections

In [ ]:
# Cell 6: Top Projected Hitters by Year

print("\nTop 25 Projected Hitters by Year:")
print("=" * 70)

for year in range(1, YEARS_AHEAD + 1):
    war_col = f'war_year_{year}'
    print(f"\nYear {year} ({BASE_YEAR + year}):")
    
    top_hitters = hitter_projections.nlargest(25, war_col)[[
        'playerid', war_col
    ]].copy()
    
    top_hitters.columns = ['Player ID', f'Year {year} WAR']
    print(top_hitters.to_string(index=False))
    print()
    print(f"  Top 25 average: {top_hitters[f'Year {year} WAR'].mean():.2f} WAR")

In [ ]:
# Cell 7: Projection Statistics by Year

print("\nHitter Projection Statistics:")
print("=" * 70)

for year in range(1, YEARS_AHEAD + 1):
    war_col = f'war_year_{year}'
    
    print(f"\nYear {year} ({BASE_YEAR + year}):")
    print(f"  Total WAR: {hitter_projections[war_col].sum():.1f}")
    print(f"  Mean WAR: {hitter_projections[war_col].mean():.2f}")
    print(f"  Median WAR: {hitter_projections[war_col].median():.2f}")
    print(f"  Std Dev: {hitter_projections[war_col].std():.2f}")
    print(f"  Min WAR: {hitter_projections[war_col].min():.2f}")
    print(f"  Max WAR: {hitter_projections[war_col].max():.2f}")
    
    # Count by tier
    elite = (hitter_projections[war_col] >= 4.0).sum()
    above_avg = ((hitter_projections[war_col] >= 2.0) & (hitter_projections[war_col] < 4.0)).sum()
    average = ((hitter_projections[war_col] >= 0.0) & (hitter_projections[war_col] < 2.0)).sum()
    below_avg = (hitter_projections[war_col] < 0.0).sum()
    
    print(f"\n  Distribution:")
    print(f"    Elite (4+ WAR): {elite}")
    print(f"    Above Average (2-4 WAR): {above_avg}")
    print(f"    Average (0-2 WAR): {average}")
    print(f"    Below Average (<0 WAR): {below_avg}")

## Step 4: Position-Specific Analysis

In [ ]:
# Cell 8: Position-Specific Projections

print("\nAverage WAR by Position (Year 1):")
print("=" * 70)

if 'Position' in hitter_projections.columns or 'primary_position' in hitter_projections.columns:
    pos_col = 'Position' if 'Position' in hitter_projections.columns else 'primary_position'
    
    position_stats = hitter_projections.groupby(pos_col)['war_year_1'].agg([
        ('count', 'count'),
        ('mean', 'mean'),
        ('total', 'sum')
    ]).sort_values('mean', ascending=False)
    
    print(position_stats.to_string())
else:
    print("Position information not available in projections.")

## Step 5: Save and Export

In [ ]:
# Cell 9: Export Projections

# Projections already saved by pipeline, but can export additional formats

print("\nExporting hitter projections...")

# Save top 100 to CSV
top_100_path = project_root / f"predictions/top_100_hitters_{BASE_YEAR + 1}.csv"
top_100 = hitter_projections.nlargest(100, 'war_year_1')
top_100.to_csv(top_100_path, index=False)
print(f"  Top 100 hitters saved to: {top_100_path}")

# Save elite hitters (4+ WAR)
elite_path = project_root / f"predictions/elite_hitters_{BASE_YEAR + 1}.csv"
elite_hitters = hitter_projections[hitter_projections['war_year_1'] >= 4.0]
elite_hitters.to_csv(elite_path, index=False)
print(f"  Elite hitters (4+ WAR) saved to: {elite_path}")
print(f"    Count: {len(elite_hitters)}")

print("\nExport complete!")

## Summary

Hitter projections generated successfully!

**Output Files:**
- `predictions/future_projections_hitter_YYYY.csv` - All hitter projections
- `predictions/top_100_hitters_YYYY.csv` - Top 100 projected hitters
- `predictions/elite_hitters_YYYY.csv` - Elite (4+ WAR) hitters

**Next Steps:**
- Run pitcher projections: See `pitcher_future_projections.ipynb`
- Validate projections: See `sWARm_future_deep_dive.ipynb`
- Combine with pitcher projections for league-wide zero-sum constraint